In [42]:
import numpy as np
import pandas as pd

In [43]:
cases = pd.read_csv('data100k.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data100k.csv'

In [ ]:
cases.head(3).T

,0,1,2
person_id,102090000000110,343221000000125,343221000000125
HearingDate,2019-02-28,2009-12-07,2011-01-20
CodeSection,A.46.2-862,B.46.2-301,A.46.2-707
codesection,covered elsewhere,covered elsewhere,covered elsewhere
ChargeType,Misdemeanor,Misdemeanor,Misdemeanor
chargetype,Misdemeanor,Misdemeanor,Misdemeanor
Class,1,1,3
DispositionCode,Guilty,Guilty,Guilty
disposition,Conviction,Conviction,Conviction
Plea,NaN,NaN,NaN


## What code sections are most frequent?


In [ ]:
cases['CodeSection'].value_counts().head(15)

CodeSection
A.46.2-862    26379
B.46.2-301    25967
46.2-300      17934
C.46.2-862    11728
18.2-250.1    10573
A.18.2-266     8568
18.2-95        7561
18.2-250       6949
18.2-57        6699
A.46.2-852     6667
18.2-388       6530
18.2-248       5632
18.2-96        5546
18.2-172       5423
19.2-306       4277
Name: count, dtype: int64


## Which ones most often lead to convictions?


In [ ]:
cases['DispositionCode'].value_counts()

DispositionCode
Guilty                     156563
Nolle Prosequi              54680
Dismissed                   42520
Guilty In Absentia          31958
Not Guilty                   5807
Not Guilty/Acquitted         1623
Not True Bill                 250
No Indictment Presented       178
Dismissed/Other                19
Name: count, dtype: int64

In [ ]:
cases['conviction'] = [x in ['Guilty', 'Guilty In Absentia'] for x in cases['DispositionCode']]
cases['conviction'].value_counts()

conviction
True     188521
False    105077
Name: count, dtype: int64

In [ ]:
cases_convict_rate = cases.groupby('CodeSection').agg({'conviction':['mean', 'count']}).reset_index()
cases_convict_rate.columns = ['CodeSection', 'convict_rate', 'count']
cases_convict_rate = cases_convict_rate.query("count >= 30")
cases_convict_rate.sort_values('convict_rate', ascending=False)

,CodeSection,convict_rate,count
1806,23-55,0.981818,55
1633,21-336,0.960000,50
1755,23-22.1(A),0.954198,131
2103,29-17(C),0.942857,70
4111,G.18.2-266,0.930233,43
...,...,...,...
2321,3.2-6503.1,0.071429,42
140,11.1-2,0.052632,38
2481,35-416,0.033333,30
1433,19.2-100,0.000000,238



## Which ones have the most severe racial disparities?


In [ ]:
cases['Race'].value_counts()

Race
White Caucasian(Non-Hispanic)                  114421
Black(Non-Hispanic)                             80173
White Caucasian (Non-Hispanic)                  41679
Black (Non-Hispanic)                            33254
Hispanic                                         9319
White                                            3527
Other(Includes Not Applicable.. Unknown)         3452
Asian Or Pacific Islander                        2787
Black                                            2200
MISSING                                          1022
Unknown (Includes Not Applicable.. Unknown)       785
Other (Includes Not Applicable.. Unknown)         615
American Indian                                   302
Unknown                                            54
Asian or Pacific Islander                           7
American Indian Or Alaskan Native                   1
Name: count, dtype: int64

In [ ]:
cases['Race'].unique()

<StringArray>
[                        'Black(Non-Hispanic)',
                                    'Hispanic',
               'White Caucasian(Non-Hispanic)',
                                     'MISSING',
                   'Asian Or Pacific Islander',
                        'Black (Non-Hispanic)',
              'White Caucasian (Non-Hispanic)',
    'Other(Includes Not Applicable.. Unknown)',
   'Other (Includes Not Applicable.. Unknown)',
                                       'Black',
                                       'White',
 'Unknown (Includes Not Applicable.. Unknown)',
                             'American Indian',
                                     'Unknown',
                   'Asian or Pacific Islander',
           'American Indian Or Alaskan Native']
Length: 16, dtype: str

In [ ]:
replace_map = {'Black(Non-Hispanic)':'Black',
                'Hispanic':'Hispanic',
                'White Caucasian(Non-Hispanic)':'White',
                'MISSING':'Other or Missing',
                'Asian Or Pacific Islander':'Asian or Pacific Islander',
                'Black (Non-Hispanic)':'Black',
                'White Caucasian (Non-Hispanic)':'White',
                'Other(Includes Not Applicable.. Unknown)':'Other or Missing',
                'Other (Includes Not Applicable.. Unknown)':'Other or Missing',
                'Black':'Black',
                'White':'White',
                'Unknown (Includes Not Applicable.. Unknown)':'Other or Missing',
                'American Indian':'American Indian or Alaskan Native',
                'Unknown':'Other or Missing',
                'Asian or Pacific Islander':'Asian or Pacific Islander',
                'American Indian Or Alaskan Native':'American Indian or Alaskan Native'}
cases['Race'] = cases['Race'].replace(replace_map)
cases['Race'].value_counts()

Race
White                                159627
Black                                115627
Hispanic                               9319
Other or Missing                       5928
Asian or Pacific Islander              2794
American Indian or Alaskan Native       303
Name: count, dtype: int64

In [ ]:
cases.groupby('Race')['conviction'].mean()

Race
American Indian or Alaskan Native    0.785479
Asian or Pacific Islander            0.662491
Black                                0.632638
Hispanic                             0.830347
Other or Missing                     0.739879
White                                0.633715
Name: conviction, dtype: float64

In [ ]:
cases.groupby('Race').agg({'conviction': 'mean'})

,conviction
Race,
American Indian or Alaskan Native,0.785479
Asian or Pacific Islander,0.662491
Black,0.632638
Hispanic,0.830347
Other or Missing,0.739879
White,0.633715



## In what localities (fips) are these disparities most severe?


In [45]:
census_race_url = 'https://virginia.box.com/shared/static/i8i5onrkveks849pkky0gwgxlax8d8fe.xlsx'

census_hisp_url = 'https://virginia.box.com/shared/static/fegrn0p0igzl95snji3ku6edwu0hy3dj.xlsx'

census_race = pd.read_excel(census_race_url, skiprows=[0,1,2,3,5,6,7])

census_race

/Users/jonathanorellana/miniconda3/envs/ds6001/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: '2021 Population Estimates'!$5:$141.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


,FIPS,Jurisdiction,Total Population,American Indian,Unnamed: 4,Asian,Unnamed: 6,Black,Unnamed: 8,Pacific Islander,Unnamed: 10,Two or more races,Unnamed: 12,White,Unnamed: 14
0,1,Accomack County,33246,441,0.013265,370,0.011129,9859,0.296547,79,0.002376,609,0.018318,23125,0.695572
1,3,Albemarle County,113535,1069,0.009416,7925,0.069802,12581,0.110812,242,0.002132,3210,0.028273,95210,0.838596
2,5,Alleghany County,14986,126,0.008408,92,0.006139,906,0.060456,17,0.001134,283,0.018884,14136,0.943280
3,7,Amelia County,13268,177,0.013340,138,0.010401,2759,0.207944,15,0.001131,259,0.019521,10445,0.787232
4,9,Amherst County,31273,495,0.015828,339,0.010840,6475,0.207048,55,0.001759,828,0.026477,24796,0.792888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,800,Suffolk city,96194,1052,0.010936,3115,0.032382,43068,0.447720,232,0.002412,2975,0.030927,51977,0.540335
129,810,Virginia Beach city,457672,6241,0.013636,44491,0.097212,104827,0.229044,1846,0.004033,20837,0.045528,324018,0.707970
130,820,Waynesboro city,22550,309,0.013703,522,0.023149,3665,0.162528,35,0.001552,792,0.035122,18840,0.835477
131,830,Williamsburg city,15590,202,0.012957,1164,0.074663,2754,0.176652,37,0.002373,591,0.037909,12055,0.773252


In [46]:
census_race.columns

Index(['FIPS', 'Jurisdiction', 'Total Population', 'American Indian',
       'Unnamed: 4', 'Asian ', 'Unnamed: 6', 'Black', 'Unnamed: 8',
       'Pacific Islander', 'Unnamed: 10', 'Two or more races', 'Unnamed: 12',
       'White ', 'Unnamed: 14'],
      dtype='object')

In [47]:
census_race = census_race[['FIPS', 'Jurisdiction', 'Total Population','American Indian', 'Asian ', 'Black', 'Pacific Islander','White ']]

census_race['Asian or Pacific Islander'] = census_race['Asian '] + census_race['Pacific Islander']
census_race = census_race.drop(['Asian ', 'Pacific Islander'], axis=1)
census_race.rename({'White ':'White'}, axis=1)


/var/folders/c5/26w6_6bj1tndp8sxfdckrkc80000gn/T/ipykernel_51521/937567107.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  census_race['Asian or Pacific Islander'] = census_race['Asian '] + census_race['Pacific Islander']


,FIPS,Jurisdiction,Total Population,American Indian,Black,White,Asian or Pacific Islander
0,1,Accomack County,33246,441,9859,23125,449
1,3,Albemarle County,113535,1069,12581,95210,8167
2,5,Alleghany County,14986,126,906,14136,109
3,7,Amelia County,13268,177,2759,10445,153
4,9,Amherst County,31273,495,6475,24796,394
...,...,...,...,...,...,...,...
128,800,Suffolk city,96194,1052,43068,51977,3347
129,810,Virginia Beach city,457672,6241,104827,324018,46337
130,820,Waynesboro city,22550,309,3665,18840,557
131,830,Williamsburg city,15590,202,2754,12055,1201


In [48]:
census_hisp = pd.read_excel(census_hisp_url, skiprows=[0,1,2,3,5,6,7,8,9])
census_hisp = census_hisp[['FIPS', 'Unnamed: 6']]
census_hisp = census_hisp.rename({'Unnamed: 6':'Hispanic'}, axis=1)
census_hisp

/Users/jonathanorellana/miniconda3/envs/ds6001/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: '2021 Population Estimates'!$5:$143.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


,FIPS,Hispanic
0,1,3170
1,3,6750
2,5,265
3,7,507
4,9,849
...,...,...
128,800,4684
129,810,40525
130,820,2244
131,830,1183


In [49]:
census = pd.merge(census_race, census_hisp, 
                  on='FIPS', 
                  how='outer',
                  validate='one_to_one',
                  indicator='matched')

In [50]:
census['matched'].value_counts()

matched
both          133
left_only       0
right_only      0
Name: count, dtype: int64

In [51]:
census = census.drop('matched', axis=1)
census

,FIPS,Jurisdiction,Total Population,American Indian,Black,White,Asian or Pacific Islander,Hispanic
0,1,Accomack County,33246,441,9859,23125,449,3170
1,3,Albemarle County,113535,1069,12581,95210,8167,6750
2,5,Alleghany County,14986,126,906,14136,109,265
3,7,Amelia County,13268,177,2759,10445,153,507
4,9,Amherst County,31273,495,6475,24796,394,849
...,...,...,...,...,...,...,...,...
128,800,Suffolk city,96194,1052,43068,51977,3347,4684
129,810,Virginia Beach city,457672,6241,104827,324018,46337,40525
130,820,Waynesboro city,22550,309,3665,18840,557,2244
131,830,Williamsburg city,15590,202,2754,12055,1201,1183


In [ ]:
cases.head(5).T

NameError: name 'cases' is not defined

Research Question: For which geohraphic locations and ehich code sections are there biggest dispartirirs bettewnn convicted of white and black defenders relatice to their censes popolation?

In [ ]:
cases_convict = cases.query("conviction == True")


In [ ]:
cases_convict = cases_convict.groupby(['CodeSection', 'Race']).size().reset_index(name='count')

In [ ]:
cases_convoct = cases_convict.rename({0:'convictions'}, axis=1)

In [53]:
cases_wide = cases_wide.pivot(index=['CodeSection', 'fips'], 
                              columns='Race', 
                              values=['convictions']), fill_value=0).reset_index()

cases_wide.columns = ['CodeSection', 'fips', 'Asian or Pacific Islander', 'Black', 'Hispanic', 'Other or Missing', 'White']
cases_wide

SyntaxError: unmatched ')' (2723535280.py, line 3)

X = cases_wide 
Y = census


In [ ]:
cases_census = pd.merge(cases_wide, census,
                        left_on='fips',
                        right_on='FIPS',
                        how='outer',
                        validate='many_to_one',
                        indicator='matched')

In [ ]:
cases_census['matched'].value_counts()

In [ ]:
cases_census.query("matched == 'left_only'")['fips'].value_counts

In [ ]:
cases_census.query("matched == 'right_only'")['fips'].value_counts

In [ ]:
cases_census['disparity_index'] = (cases_census['Black'] / cases_census['Black'].sum()) / (cases_census['Black Population'] / cases_census['Black Population'].sum())

In [ ]:
(cases_census['Black'] / cases_census['Black'].sum()) / (cases_census['Black Population'] / cases_census['Black Population'].sum())/(cases_census['White'] / cases_census['White'].sum())